# Xplore: **network**

In [ ]:
######################################## Parameters

rule = 'solve_network'                        # results/networks/solved_{horizon}.nc


### Run (folders resources/<prefix>/<name>/ and results/<prefix>/<name>/)
name = ''
prefix = ''

### Planning horizon (only used by compose/solve networks)
horizon = ''

### Spatial domain tag for the map boundaries (coordinates defined in params.yaml, e.g. 'ES', 'EU').
spatial_domain = 'ES' 


In [ ]:
##### Import packages
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import sys


##### Import local functions.
##### Dropping the cached 'functions.*' modules first makes re-running this cell pick up
##### any edit made in functions/ without restarting the kernel (and reloading the
##### network). Note this only happens when this cell is executed, not on every cell.
sys.path.append(os.path.abspath(os.path.join("..")))
for _module in [m for m in list(sys.modules) if m == 'functions' or m.startswith('functions.')]:
    del sys.modules[_module]
import functions as xp


##### Read params.yaml
params = xp.read_params("../params.yaml")


##### Map boundaries for the selected spatial domain (from params.yaml)
boundaries_onshore = params[f'boundaries_onshore_{spatial_domain}']
boundaries_offshore = params[f'boundaries_offshore_{spatial_domain}']


##### Ignore warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

Load the network, its region files and auto-detect the available sections. Show the network components.

In [ ]:
##### Resolve the network source for the selected rule
src = xp.resolve_network_source(rule, horizon=horizon)
print(f"rule     : {rule}")
print(f"file     : {src['filename']}")
print(f"location : {src['location']}")
print(f"regions  : {src['regions_onshore']} / {src['regions_offshore']}")

##### Load the network
n = xp.load_network(
    params,
    src['filename'],
    prefix=prefix,
    name=name,
    location=src['location'],
)

##### Region files (onshore / offshore) matching this rule
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    filenames=(src['regions_onshore'], src['regions_offshore']),
)

##### Auto-detect which sections make sense for this network state
sections = xp.detect_sections(n)


##### Network components
n

Sections available for this network state:

In [ ]:
pd.Series(sections, name='available').to_frame()

# Analyses

The following analyses only apply to a **solved** network (`solve_network` or `solve_operations_network`).

## Analysis: Electric capacity expansion (at AC and low-voltage buses)

Initial (`p_nom`) and optimal (`p_nom_opt`) **electric capacity at the AC and low-voltage buses** of every generator, storage unit and link, from `n.statistics.installed_capacity` / `n.statistics.optimal_capacity` with `bus_carrier=['AC', 'low voltage']`.

Each component is measured at its port on an electric bus. For a link this is `p_nom × efficiency` of that port, since `p_nom` refers to its input (`bus0`): e.g. a `CCGT` (gas → AC) counts its electric output.

- **Positive**: capacity that supplies the electric buses (generators, `solar rooftop`, `CCGT`, battery discharge, `V2G`, imports).
- **Negative**: capacity that withdraws from them (`H2 Electrolysis`, battery and `BEV charger`, heat pumps, exports, pipeline compression).

**Heat pumps** are defined in reverse (`bus0` = heat, `bus1` = low voltage, `p_nom` in MW of heat) with a time-varying efficiency (`1/COP`), which `n.statistics` does not use. Their electric capacity is computed as `p_nom × max(efficiency)`, the peak electric power they can draw.

Links between two electric buses (DC lines, distribution grid) are left out: their net value at the electric ports is only losses or flows in opposite directions (see *Grid expansion*). Capacity at low voltage sits behind the distribution grid.

In [ ]:
if sections['solved']:
    #################### Parameters
    units = 'GW'               # MW | GW | TW

    #################### Table + bar plot
    display(xp.summary_electric_capacities(n, units=units))
else:
    xp.notify_skipped('Analysis: Electric capacity expansion')

## Analysis: System costs

Annualised capital cost (`n.statistics.capex`) and operational cost (`n.statistics.opex`) per component and carrier. Their sum is `n.statistics.system_cost`.

It differs from the optimisation objective (`n.objective`) by the capital cost of the capacity that already existed (`n.statistics.installed_capex`), which the optimiser cannot change and therefore leaves out. Negative values are revenues (e.g. electricity exports).

In [ ]:
if sections['solved']:
    #################### Parameters
    units = 'bnEUR'            # EUR | MEUR | bnEUR
    top = 15                   # carriers shown in the bar plot

    #################### Table + bar plot
    display(xp.summary_system_costs(n, units=units, top=top))
else:
    xp.notify_skipped('Analysis: System costs')

## Analysis: Grid expansion

Where was grid expansion required? Plot the increase of line/link capacities.

In [ ]:
if sections['solved']:
    line_widths = 1 * (n.lines.s_nom_opt - n.lines.s_nom) / 1e3
    link_widths = 1 * (n.links.p_nom_opt - n.links.p_nom) / 1e3
    xp.plot_network_map(
        n, gdf_regions_onshore, gdf_regions_offshore, params, boundaries,
        line_widths=line_widths, link_widths=link_widths,
    )
else:
    xp.notify_skipped('Analysis: Grid expansion')

## Analysis: Grid congestion

Where are the bottlenecks in the grid from the optimal dispatch?

In [ ]:
if sections['solved']:
    #################### Parameters
    cong_criterion = ('quantile', 0.9)   # or ('mean', 'na')

    line_widths = 1 * (n.lines.s_nom_opt) / 1e3
    link_widths = 0 * (n.links.p_nom_opt) / 1e3   # WIP: link congestion
    lnt_p0 = n.lines_t['p0']

    #################### Line colors (loading relative to the optimal capacity s_nom_opt)
    if cong_criterion[0] == 'mean':
        ln_result_criterion = lnt_p0.abs().mean() / n.lines['s_nom_opt']
        title_cb = 'Mean of line CF'
    else:  # 'quantile'
        ln_result_criterion = lnt_p0.abs().quantile(cong_criterion[1]) / n.lines['s_nom_opt']
        title_cb = f'Q{cong_criterion[1]} of line CF'

    norm = mcolors.Normalize(vmin=0, vmax=0.7)
    cmap = plt.cm.rainbow
    ln_colors = [cmap(norm(p)) for p in ln_result_criterion]

    #################### Figure
    fig, ax = plt.subplots(figsize=[12, 12], subplot_kw={'projection': ccrs.PlateCarree()})
    n.plot(ax=ax, geomap=False, line_widths=line_widths, link_widths=link_widths,
           line_colors=ln_colors, bus_sizes=params['bus_sizes'],
           bus_colors=params['bus_colors'], boundaries=boundaries)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical', shrink=0.55, pad=0.02)
    cbar.set_label(title_cb, fontsize=12)
    xp.map_add_region(ax, gdf_regions_onshore, params['map_add_region'])
    xp.map_add_features(ax, params['map_add_features'])
else:
    xp.notify_skipped('Analysis: Grid congestion')

## Analysis: Interconnections

Electricity imports/exports through the border links (pypsa-spain interconnections, carriers `DC_ic export` / `DC_ic import`).

`n.statistics.energy_balance(bus_carrier='AC', groupby=False)` gives, for each link, the energy at its port on a Spanish AC bus, weighted by the snapshot duration: negative for export links (they withdraw from the AC bus), positive for import links (they inject into it).

In [ ]:
if sections['interconnections']:
    #################### Parameters
    units = 'TWh'              # MWh | GWh | TWh

    #################### Bar plot
    xp.plot_interconnections_energy(n, units=units)
else:
    xp.notify_skipped('Analysis: Interconnections')

Interconnection power time series (exports positive, imports negative). With `aggregate_time=False`, `energy_balance` returns the power [MW] at each snapshot, not weighted.

In [ ]:
if sections['interconnections']:
    #################### Parameters
    start = '2019-03-01'
    end = '2019-04-01'

    #################### Time series plot
    xp.plot_interconnections_timeseries(n, start=start, end=end)
else:
    xp.notify_skipped('Analysis: Interconnections (time series)')

Heatmap of the net exchange with each neighbouring country (sum of all its border links), days on the horizontal axis and hours on the vertical one: exports positive (green), imports negative (pink). The colour scale is centred at 0. If the snapshots span more than one hour, each one is placed at its starting hour.

In [ ]:
if sections['interconnections']:
    #################### Parameters
    countries = ['FR', 'PT']
    tol = 1e-3                 # [GW] |net| below this counts as zero (solver residuals)

    #################### Heatmaps (one per country)
    xp.plot_interconnections_heatmap(n, countries=countries, tol=tol)
else:
    xp.notify_skipped('Analysis: Interconnections (heatmaps)')

Economic balance of the trade with each country. Imports and exports are priced by the market generators at the border buses (`DC_ic market export`, sign -1 and `marginal_cost` = -(export price); `DC_ic market import`, `marginal_cost` = import price), with the same prices for all the nodes of a country. Energy is measured at the border bus (after the link losses), as in the objective function.

Histograms of the energy traded per price bin (dashed lines: energy-weighted average prices). Title: balance = export revenue - import cost. This is only the cash flow with the neighbouring market, not a welfare measure (it does not include the value of that energy inside Spain).

In [ ]:
if sections['interconnections']:
    #################### Parameters
    countries = ['FR', 'PT']
    bin_width = 10             # [€/MWh] price bin of the histograms
    price_cut = 0.99           # x axis up to this quantile of the traded energy (long price tails)

    #################### Table + histograms (one per country)
    display(xp.summary_interconnections_trade(
        n, countries=countries, bin_width=bin_width, price_cut=price_cut,
    ))
else:
    xp.notify_skipped('Analysis: Interconnections (economic balance)')

Gain from trade with each country, valuing the exchanged energy at the Spanish nodal price (`n.buses_t.marginal_price`) of the AC bus each border link is connected to:

- export: market revenue - value at the Spanish price of the energy that leaves Spain (before the link losses)
- import: value at the Spanish price of the energy that reaches Spain (after the link losses) - market cost

Nodal prices are marginal values, so this is the gain of each traded MWh against producing (import) or using (export) it at the margin inside Spain. In an optimal dispatch it is only positive when the link is at full capacity, i.e. it is the congestion rent of the interconnections, the value of their capacity. Left: Spanish vs foreign price of each traded hour (exports above the diagonal, imports below). Right: monthly gain.

In [ ]:
if sections['interconnections']:
    #################### Parameters
    countries = ['FR', 'PT']
    price_cut = 0.99           # axes up to this quantile of the prices (long price tails)
    tol = 1                    # [MW] hours with less traded power are not plotted in the scatter

    #################### Table + scatter and monthly gain (one per country)
    display(xp.summary_interconnections_gain(
        n, countries=countries, price_cut=price_cut, tol=tol,
    ))
else:
    xp.notify_skipped('Analysis: Interconnections (gain from trade)')